# Mini-TP 4 — Puntúa tu modelo sobre un flujo (starter)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Entrega individual, esta semana**

Aplica **tu modelo** (el de la Sesión 1) sobre un **flujo de eventos** en lugar de pedidos sueltos. Las celdas de la actividad fueron completadas en esta versión.

**Qué debes entregar**
1. Un **flujo** de eventos con las *features* de tu modelo (simulado, o consumido de Kafka).
2. Un **consumidor** que puntúe tu modelo sobre **cada** evento (inferencia online).
3. **Métricas por ventana:** throughput (ev/s), latencia **p95** y un indicador simple de **drift** de entradas.
4. Una **alerta** cuando una métrica cruce un umbral.
5. Una **comparación** contra puntuar el mismo lote en **batch**, con una breve reflexión.

**Se evalúa:** que corra de punta a punta; inferencia online sobre el flujo; métricas por ventana correctas; y la reflexión streaming vs batch.

> Apóyate en `streaming_tutorial.ipynb`. Entorno uv: `uv add scikit-learn joblib numpy kafka-python`.


In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "joblib", "numpy"])
print("Listo.")

Listo.


## 1. Tu modelo

Carga **tu** modelo. Si aún no lo tienes a mano, el placeholder deja probar la mecánica.

In [2]:
import joblib, numpy as np
from sklearn.linear_model import LogisticRegression

# Para que el notebook sea autocontenido se usa un modelo de ejemplo reproducible.
# Si se dispone del modelo de la Sesión 1, basta con reemplazar este bloque por:
# modelo = joblib.load("model.pkl")
# N_FEATURES = modelo.n_features_in_
rng_modelo = np.random.RandomState(0)
_X = rng_modelo.randn(300, 3)
_y = (_X.sum(axis=1) > 0).astype(int)
modelo = LogisticRegression(random_state=0).fit(_X, _y)
N_FEATURES = modelo.n_features_in_

print("Modelo cargado. n_features =", N_FEATURES)


Modelo cargado. n_features = 3


## 2. El flujo

Simula un flujo con las features de tu modelo. Introduce un cambio de distribución a mitad de camino para poder detectar drift. (Alternativa: consumir de Kafka como en el tutorial.)

In [3]:
import queue, threading, time, random

stream = queue.Queue()
STOP = object()
N = 800
rng_stream = random.Random(0)
eventos_generados = []

def productor():
    for i in range(N):
        # Primera mitad ~ N(0,1); segunda mitad ~ N(1.2,1) para simular drift.
        mu = 0.0 if i < N // 2 else 1.2
        values = [rng_stream.gauss(mu, 1.0) for _ in range(N_FEATURES)]
        ev = {"values": values, "t": time.time()}
        eventos_generados.append(ev.copy())
        stream.put(ev)
        time.sleep(0.0005)
    stream.put(STOP)

print("Productor listo.")


Productor listo.


## 3. Consumidor con inferencia online + métricas por ventana

In [4]:
from collections import deque
import numpy as np

latencias = []
ventana = deque()
alertas = []
metricas_ventana = []

# El modelo de referencia fue entrenado con features centradas cerca de 0 y desvío ~1.
BASELINE_MEDIA_F0 = float(np.mean(_X[:, 0]))
BASELINE_STD_F0 = float(np.std(_X[:, 0]))
UMBRAL_DRIFT = 0.50  # desplazamiento de la media expresado en desvíos estándar

def consumidor():
    procesados = 0
    while True:
        ev = stream.get()
        if ev is STOP:
            break

        t0_pred = time.perf_counter()
        pred = modelo.predict([ev["values"]])[0]
        lat_ms = (time.perf_counter() - t0_pred) * 1000
        latencias.append(lat_ms)

        ahora = ev["t"]
        ventana.append((ahora, ev["values"][0], pred, lat_ms))
        while ventana and ahora - ventana[0][0] > 1.0:  # ventana deslizante de 1 s
            ventana.popleft()

        procesados += 1
        if procesados % 200 == 0:
            valores_f0 = np.array([w[1] for w in ventana], dtype=float)
            lats = np.array([w[3] for w in ventana], dtype=float)
            media_f0 = float(valores_f0.mean())
            drift_score = abs(media_f0 - BASELINE_MEDIA_F0) / max(BASELINE_STD_F0, 1e-9)

            if len(ventana) > 1:
                ancho = max(ventana[-1][0] - ventana[0][0], 1e-9)
                throughput_ventana = (len(ventana) - 1) / ancho
            else:
                throughput_ventana = 0.0

            metrica = {
                "procesados": procesados,
                "eventos_ventana": len(ventana),
                "throughput_ev_s": throughput_ventana,
                "p95_ms": float(np.percentile(lats, 95)),
                "media_f0": media_f0,
                "drift_score": drift_score,
            }
            metricas_ventana.append(metrica)

            if drift_score > UMBRAL_DRIFT:
                alertas.append((procesados, drift_score, media_f0))

t0 = time.time()
pt = threading.Thread(target=productor)
ct = threading.Thread(target=consumidor)
pt.start(); ct.start(); pt.join(); ct.join()
dur = time.time() - t0

print(f"throughput global : {len(latencias)/dur:.0f} ev/s")
print(f"p95 global        : {np.percentile(latencias, 95):.3f} ms")
print()
print("Métricas por ventana:")
for m in metricas_ventana:
    print(
        f"  n={m['procesados']:3d} | ventana={m['eventos_ventana']:3d} | "
        f"throughput={m['throughput_ev_s']:.0f} ev/s | p95={m['p95_ms']:.3f} ms | "
        f"drift={m['drift_score']:.2f}"
    )
print()
print("Alertas de drift:", alertas)


throughput global : 1676 ev/s
p95 global        : 0.137 ms

Métricas por ventana:
  n=200 | ventana=200 | throughput=1675 ev/s | p95=0.123 ms | drift=0.05
  n=400 | ventana=400 | throughput=1681 ev/s | p95=0.132 ms | drift=0.02
  n=600 | ventana=600 | throughput=1676 ev/s | p95=0.138 ms | drift=0.37
  n=800 | ventana=800 | throughput=1679 ev/s | p95=0.137 ms | drift=0.58

Alertas de drift: [(800, 0.5840367609227655, 0.5884572250138776)]


## 4. Comparación contra batch

Puntúa el mismo conjunto de eventos de una sola vez (batch) y compara tiempo/uso frente al streaming.

In [5]:
# Se puntúan en batch exactamente los mismos vectores que atravesaron el stream.
LOTE = np.asarray([ev["values"] for ev in eventos_generados], dtype=float)

t0_batch = time.perf_counter()
batch_pred = modelo.predict(LOTE)
batch_total_ms = (time.perf_counter() - t0_batch) * 1000
batch_por_evento_ms = batch_total_ms / len(LOTE)

stream_promedio_ms = float(np.mean(latencias))
stream_p95_ms = float(np.percentile(latencias, 95))

print(f"Eventos                    : {len(LOTE)}")
print(f"Batch total                : {batch_total_ms:.3f} ms")
print(f"Batch por evento (promedio): {batch_por_evento_ms:.6f} ms")
print(f"Streaming por evento (mean): {stream_promedio_ms:.6f} ms")
print(f"Streaming por evento (p95) : {stream_p95_ms:.6f} ms")
print(f"Predicciones batch positivas: {int(batch_pred.sum())}")


Eventos                    : 800
Batch total                : 0.439 ms
Batch por evento (promedio): 0.000548 ms
Streaming por evento (mean): 0.087639 ms
Streaming por evento (p95) : 0.137057 ms
Predicciones batch positivas: 597


### Reflexión

El **streaming** permite obtener una predicción apenas llega cada evento y, al mismo tiempo, monitorear métricas recientes mediante ventanas deslizantes. En este ejercicio el drift se detectó comparando la media de la primera feature contra la distribución de referencia del entrenamiento y generando una alerta cuando el desplazamiento superó el umbral definido. El **batch** es más eficiente por evento porque vectoriza el cálculo, pero el resultado sólo está disponible cuando se procesa el lote. Usaría streaming cuando la frescura de la predicción sea importante y batch para procesos masivos u offline. Ante una alerta de drift, primero validaría la calidad de los datos y el desempeño del modelo; si el cambio persiste, dispararía monitoreo/reentrenamiento y mantendría una versión estable como fallback.


## 5. (Opcional) Con Kafka en Docker

Levanta un broker y reemplaza la cola por un topic real:
```bash
docker run -d --name redpanda -p 9092:9092 \
  redpandadata/redpanda redpanda start --overprovisioned --smp 1 --check=false
uv add kafka-python
```
Produce a un topic `eventos` y consume puntuando tu modelo (ver la parte 6 del tutorial).